# Paddy disease prediction

Train a MobileNetV2 classifier to recognize the categories in your labeled paddy leaf folders. The workflow deduplicates JPEG and PNG inputs, creates fixed training, validation, and test sets, and exports a Keras model with its class order and evaluation report.

Use Python 3.12 with TensorFlow, NumPy, Pillow, Matplotlib, and scikit-learn. For installation and dataset layout, see the repository README. Set `DATA_DIR` in the next cell, then run the notebook from top to bottom.

In [ ]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = Path("data/train_images")
KAGGLE_TRAIN_CSV = None  # Set to DATA_DIR.parent / "train.csv" for the original Paddy Doctor photos.
EXTRA_DATA_DIRS = ()
HOLDOUT_DIR = None  # Set to a separately collected labeled test folder when available.
GROUP_CSV = None  # Optional CSV with path,group columns for known related images.
RUN_DIR = Path("runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

SEED = 42
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 8
FINE_TUNE_LAYERS = 30
BACKBONE_WEIGHTS = "imagenet"
CACHE_FROZEN_BACKBONE = False  # Faster CPU training when running from the full repository.

## Inventory and fixed splits

Point `DATA_DIR` and any optional entries in `EXTRA_DATA_DIRS` at directories with class subfolders. For a separately collected test set, point `HOLDOUT_DIR` at another directory with the same class folders. Leave the archive's overlapping labeled test folder out of the evaluation.

Use a `path,group` CSV to keep photos from the same plant, capture session, or known near-duplicate group together. Relative paths are resolved from the CSV location. Exact duplicate detection hashes decoded RGB pixels. It cannot detect crops or recompressed variants with different pixels. The splitter rejects conflicting labels and exact or declared group overlap with an external test set.

In [ ]:
import csv
import hashlib
from collections import Counter, defaultdict

from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def find_labeled_images(root):
    """Read JPEG and PNG paths labeled by their top-level folder."""
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise ValueError(f"Image directory does not exist: {root}")

    records = []
    for folder in sorted(root.iterdir()):
        if not folder.is_dir() or folder.name.startswith("."):
            continue
        paths = sorted(
            path for path in folder.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            and not any(part.startswith(".") for part in path.relative_to(folder).parts)
        )
        records.extend((path.resolve(), folder.name) for path in paths)
    if not records:
        raise ValueError(f"No JPEG or PNG images in class subfolders of {root}")
    return records


def select_kaggle_originals(records, train_csv):
    """Match original image IDs and labels against the numbered competition photos."""
    expected = set()
    image_ids = set()
    with Path(train_csv).expanduser().open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not {"image_id", "label"}.issubset(reader.fieldnames or []):
            raise ValueError("The Kaggle training CSV must contain image_id and label columns.")
        for line, row in enumerate(reader, start=2):
            image_id = (row["image_id"] or "").strip()
            label = (row["label"] or "").strip()
            if not image_id or not label or image_id in image_ids:
                raise ValueError(f"Missing or repeated image ID or label on Kaggle CSV row {line}.")
            expected.add((image_id, label))
            image_ids.add(image_id)
    if not expected:
        raise ValueError("The Kaggle training CSV contains no image records.")
    selected = {}
    for path, label in records:
        key = (path.name, label)
        if key in expected:
            if key in selected:
                raise ValueError(f"More than one file matches Kaggle image {path.name} in class {label}.")
            selected[key] = (path, label)
    missing = expected - selected.keys()
    if missing:
        image_id, label = min(missing)
        raise ValueError(f"Missing {len(missing)} Kaggle originals, including {label}/{image_id}.")
    return list(selected.values())


def read_image_groups(group_csv):
    """Load optional path,group rows with paths relative to the CSV file."""
    if group_csv is None:
        return {}
    group_csv = Path(group_csv).expanduser().resolve()
    with group_csv.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not {"path", "group"}.issubset(reader.fieldnames or []):
            raise ValueError("The group CSV must contain path and group columns.")
        groups = {}
        for line, row in enumerate(reader, start=2):
            path_text = (row["path"] or "").strip()
            group = (row["group"] or "").strip()
            if not path_text or not group:
                raise ValueError(f"Empty path or group on row {line} of {group_csv}")
            path = (group_csv.parent / path_text).resolve()
            if groups.setdefault(path, group) != group:
                raise ValueError(f"Conflicting group IDs for {path}")
    return groups


def image_fingerprint(path):
    """Hash dimensions and decoded RGB pixels without applying EXIF rotation."""
    try:
        with Image.open(path) as image:
            if image.format not in {"JPEG", "PNG"}:
                raise ValueError(f"Expected JPEG or PNG content in {path}")
            rgb = image.convert("RGB")
            digest = hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode("ascii"))
            digest.update(rgb.tobytes())
    except (OSError, Image.DecompressionBombError) as error:
        raise ValueError(f"Could not decode image: {path}") from error
    return digest.hexdigest()


def group_components(fingerprints, metadata):
    """Put linked fingerprints in one group, including links through duplicates."""
    parents = {fingerprint: fingerprint for fingerprint in fingerprints}
    first_fingerprint = {}

    def representative(fingerprint):
        while fingerprint != parents[fingerprint]:
            parents[fingerprint] = parents[parents[fingerprint]]
            fingerprint = parents[fingerprint]
        return fingerprint

    for fingerprint, group in metadata:
        previous = first_fingerprint.setdefault(group, fingerprint)
        parents[representative(fingerprint)] = representative(previous)
    return {fingerprint: representative(fingerprint) for fingerprint in fingerprints}


def choose_holdout(records, fraction, minimum_remaining, seed):
    """Return remaining and holdout items while keeping every class in both."""
    by_class = defaultdict(list)
    for record in records:
        by_class[record[1]].append(record)

    # With no multi-image groups, sample directly within each class.
    if len({record[2] for record in records}) == len(records):
        rng = np.random.default_rng(seed)
        remaining, holdout = [], []
        for label, members in by_class.items():
            if len(members) <= minimum_remaining:
                raise ValueError(f"Class {label!r} needs at least {minimum_remaining + 1} distinct images.")
            count = min(len(members) - minimum_remaining, max(1, round(len(members) * fraction)))
            order = rng.permutation(len(members))
            holdout.extend(members[index] for index in order[:count])
            remaining.extend(members[index] for index in order[count:])
        return remaining, holdout

    group_counts = {label: len({record[2] for record in members}) for label, members in by_class.items()}
    for label, count in group_counts.items():
        if count <= minimum_remaining:
            raise ValueError(f"Class {label!r} needs at least {minimum_remaining + 1} independent groups.")

    # Choose the fold closest to the target class fractions while preserving groups.
    labels = np.array([record[1] for record in records])
    groups = np.array([record[2] for record in records])
    n_folds = min(round(1 / fraction), min(group_counts.values()))
    best = None
    for offset in range(4):
        folds = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed + offset)
        for keep, select in folds.split(np.zeros(len(labels)), labels, groups):
            selected_counts = Counter(labels[select])
            if set(selected_counts) != set(by_class):
                continue
            remaining_groups = defaultdict(set)
            for index in keep:
                remaining_groups[labels[index]].add(groups[index])
            if any(len(remaining_groups[label]) < minimum_remaining for label in by_class):
                continue
            error = sum(
                abs(selected_counts[label] / len(members) - fraction)
                for label, members in by_class.items()
            )
            if best is None or error < best[0]:
                best = (error, keep, select)
    if best is None:
        raise ValueError("Could not create class-complete disjoint splits using these groups.")
    return [records[index] for index in best[1]], [records[index] for index in best[2]]


def prepare_splits(data_dir, seed=42, holdout_dir=None, group_csv=None, extra_data_dirs=(),
                   kaggle_train_csv=None):
    """Return fixed image splits, score label order, and dataset audit counts."""
    primary = find_labeled_images(data_dir)
    discovered_primary_files = len(primary)
    if kaggle_train_csv is not None:
        primary = select_kaggle_originals(primary, kaggle_train_csv)
    class_names = sorted({label for _, label in primary})
    if len(class_names) < 2:
        raise ValueError("At least two labeled class subfolders are required.")
    extra = [record for root in extra_data_dirs for record in find_labeled_images(root)]
    external = find_labeled_images(holdout_dir) if holdout_dir is not None else []
    class_ids = {name: index for index, name in enumerate(class_names)}
    if {label for _, label in extra} - class_ids.keys():
        raise ValueError("Extra data contains class folders absent from the primary data.")
    if external and {label for _, label in external} != class_ids.keys():
        raise ValueError("The external test directory must contain the same classes as the primary data.")

    groups = read_image_groups(group_csv)
    unexpected = groups.keys() - {path for path, _ in primary + extra + external}
    if unexpected:
        raise ValueError(f"Group CSV contains paths outside the image inventory, such as {min(unexpected)}.")

    labels_by_digest = {}
    digest_by_path = {}
    group_links = []

    def unique_images(records):
        unique = {}
        for path, label in records:
            if path not in digest_by_path:
                digest_by_path[path] = image_fingerprint(path)
            digest = digest_by_path[path]
            if labels_by_digest.setdefault(digest, label) != label:
                raise ValueError(f"Identical pixels occur under conflicting class labels: {path}")
            unique.setdefault(digest, (path, label))
            if path in groups:
                group_links.append((digest, groups[path]))
        return unique

    development = unique_images(primary + extra)
    external_test = unique_images(external)
    if development.keys() & external_test.keys():
        raise ValueError("The external test set shares decoded images with the development data.")

    components = group_components(labels_by_digest, group_links)

    def split_records(unique):
        return [(str(path), label, components[digest]) for digest, (path, label) in unique.items()]

    development_records = split_records(development)
    if external_test:
        test = split_records(external_test)
        if {record[2] for record in development_records} & {record[2] for record in test}:
            raise ValueError("The external test set shares a group with the development data.")
        train, validation = choose_holdout(development_records, 0.1, 1, seed)
    else:
        development_records, test = choose_holdout(development_records, 0.1, 2, seed)
        train, validation = choose_holdout(development_records, 1 / 9, 1, seed + 1)

    parts = {"train": train, "validation": validation, "test": test}
    splits = {
        name: [(path, class_ids[label]) for path, label, _ in records]
        for name, records in parts.items()
    }
    audit = {
        "seed": seed,
        "primary_files": len(primary),
        "primary_files_excluded_by_kaggle_csv": discovered_primary_files - len(primary),
        "extra_files": len(extra),
        "external_test_files": len(external),
        "development_duplicates_removed": len(primary) + len(extra) - len(development),
        "external_duplicates_removed": len(external) - len(external_test),
        "group_csv_entries": len(groups),
        "external_test": bool(external_test),
        "split_class_counts": {
            name: dict(Counter(label for _, label, _ in records))
            for name, records in parts.items()
        },
    }
    return splits, class_names, audit

## Image loading

The input pipeline keeps aspect ratios when resizing and adds padding to reach 160 × 160 RGB. Labels are sorted in alphabetical folder order. Only the training dataset reshuffles. Class weights account for class imbalance using training counts.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
splits, class_names, audit = prepare_splits(
    DATA_DIR,
    seed=SEED,
    holdout_dir=HOLDOUT_DIR,
    group_csv=GROUP_CSV,
    extra_data_dirs=EXTRA_DATA_DIRS,
    kaggle_train_csv=KAGGLE_TRAIN_CSV,
)

for name, records in splits.items():
    counts = np.bincount([label for _, label in records], minlength=len(class_names))
    print(f"{name}: {len(records)} images", dict(zip(class_names, counts.tolist())))
print(json.dumps(audit, indent=2))


def load_rgb_image(path, image_size):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    return tf.image.resize_with_pad(image, *image_size)


def make_dataset(records, training=False):
    paths, labels = zip(*records)
    dataset = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        dataset = dataset.shuffle(len(records), seed=SEED, reshuffle_each_iteration=True)
    return (
        dataset.map(
            lambda path, label: (load_rgb_image(path, IMAGE_SIZE), label),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = make_dataset(splits["train"], training=True)
validation_ds = make_dataset(splits["validation"])
test_ds = make_dataset(splits["test"])
training_counts = np.bincount(
    [label for _, label in splits["train"]], minlength=len(class_names)
)
class_weights = {
    index: float(len(splits["train"]) / (len(class_names) * count))
    for index, count in enumerate(training_counts)
}

## Train and select the model

Start with ImageNet MobileNetV2 weights and a small classifier head. Select checkpoints using validation loss. Optional fine-tuning updates the upper backbone layers while keeping batch normalization frozen. Set `FINE_TUNE_EPOCHS = 0` to skip fine-tuning, or set `BACKBONE_WEIGHTS = None` to train from scratch without downloading weights.

In [ ]:
layers = tf.keras.layers
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="training_augmentation")

backbone = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights=BACKBONE_WEIGHTS,
)
backbone.trainable = BACKBONE_WEIGHTS is None

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="rgb_image")
features = augmentation(inputs)
features = layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_rgb")(features)
# Keep pretrained batch normalization statistics stable during fine-tuning.
features = backbone(features, training=False) if BACKBONE_WEIGHTS else backbone(features)
features = layers.GlobalAveragePooling2D(name="mobile_pool")(features)
features = layers.Dropout(0.2, name="mobile_dropout")(features)
outputs = layers.Dense(len(class_names), activation="softmax", name="class_scores")(features)
model = tf.keras.Model(inputs, outputs, name="paddy_mobilenet_v2")
print(f"Model parameters: {model.count_params():,}")

In [ ]:
def fit_phase(epochs, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=epochs,
        class_weight=class_weights,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )],
    ).history


def fit_standard(model_path):
    histories = {"head": fit_phase(HEAD_EPOCHS, 1e-3)}
    best_validation_loss = model.evaluate(validation_ds, verbose=0, return_dict=True)["loss"]
    if not np.isfinite(best_validation_loss):
        raise ValueError("Training did not produce a finite validation loss.")
    model.save(model_path)
    selected_phase = "head"

    if BACKBONE_WEIGHTS is not None and FINE_TUNE_EPOCHS > 0:
        if not 1 <= FINE_TUNE_LAYERS <= len(backbone.layers):
            raise ValueError("FINE_TUNE_LAYERS must be within the number of backbone layers.")
        backbone.trainable = True
        for layer in backbone.layers:
            layer.trainable = False
        for layer in backbone.layers[-FINE_TUNE_LAYERS:]:
            layer.trainable = not isinstance(layer, layers.BatchNormalization)
        histories["fine_tune"] = fit_phase(FINE_TUNE_EPOCHS, 1e-5)
        fine_tune_loss = model.evaluate(validation_ds, verbose=0, return_dict=True)["loss"]
        if np.isfinite(fine_tune_loss) and fine_tune_loss < best_validation_loss:
            candidate_path = RUN_DIR / "fine_tune.keras"
            model.save(candidate_path)
            candidate_path.replace(model_path)
            selected_phase = "fine_tune"
            best_validation_loss = fine_tune_loss
    return histories, best_validation_loss, selected_phase


RUN_DIR.mkdir(parents=True, exist_ok=False)
model_path = RUN_DIR / "model.keras"
if CACHE_FROZEN_BACKBONE:
    if BACKBONE_WEIGHTS is None:
        raise ValueError("Cached training requires pretrained MobileNetV2 weights.")
    from training.cached_backbone import fit_cached

    histories, best_validation_loss, selected_phase = fit_cached(
        model, backbone, train_ds, validation_ds,
        batch_size=BATCH_SIZE, head_epochs=HEAD_EPOCHS, fine_tune_epochs=FINE_TUNE_EPOCHS,
        fine_tune_layers=FINE_TUNE_LAYERS, class_weights=class_weights, seed=SEED,
        model_path=model_path,
    )
else:
    histories, best_validation_loss, selected_phase = fit_standard(model_path)

# Evaluate the saved checkpoint selected using validation loss alone.
model = tf.keras.models.load_model(model_path, compile=False)
print(f"Selected {selected_phase} checkpoint, validation loss {best_validation_loss:.4f}")

## Evaluate and export

Use the held-out test set once the model is selected. The report includes per-class precision, recall, F1 and support, macro F1, accuracy, and a confusion matrix. The run folder also records the split paths, class order, configuration, and training history. Report internal and separately collected field results with their respective dataset sources.

In [ ]:
true_batches = []
score_batches = []
for images, labels in test_ds:
    scores = model(images, training=False)
    tf.debugging.assert_all_finite(scores, "Non-finite prediction scores")
    true_batches.append(labels.numpy())
    score_batches.append(scores.numpy())

true_labels = np.concatenate(true_batches)
test_scores = np.concatenate(score_batches)
predicted_labels = test_scores.argmax(axis=1)
label_ids = list(range(len(class_names)))
report_options = dict(labels=label_ids, target_names=class_names, zero_division=0)
test_report = classification_report(true_labels, predicted_labels, **report_options, output_dict=True)
matrix = confusion_matrix(true_labels, predicted_labels, labels=label_ids)
print(classification_report(true_labels, predicted_labels, **report_options))

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
heatmap = ax.imshow(matrix, cmap="Blues")
ax.set(xticks=label_ids, yticks=label_ids, xticklabels=class_names, yticklabels=class_names,
       xlabel="Predicted class", ylabel="True class", title="Held-out test confusion matrix")
ax.tick_params(axis="x", labelrotation=45)
fig.colorbar(heatmap, ax=ax)
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=160)
plt.show()

manifest = {
    name: [{"path": path, "label": label} for path, label in records]
    for name, records in splits.items()
}
run_config = {
    "class_names": class_names,
    "image_size": list(IMAGE_SIZE),
    "pixel_range": [0, 255],
    "resize": "bilinear with aspect-ratio-preserving padding",
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "backbone_weights": BACKBONE_WEIGHTS,
    "head_epochs": HEAD_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "fine_tune_layers": FINE_TUNE_LAYERS,
    "class_weights": class_weights,
    "training_mode": "cached MobileNetV2 with training-only horizontal flips" if CACHE_FROZEN_BACKBONE else "standard",
    "dataset_filter": "train.csv original photos only" if KAGGLE_TRAIN_CSV is not None else None,
    "selected_phase": selected_phase,
    "selected_validation_loss": best_validation_loss,
    "test_source": "external holdout" if HOLDOUT_DIR is not None else "internal holdout",
    "versions": {"python": platform.python_version(), "tensorflow": tf.__version__,
                 "keras": tf.keras.__version__, "scikit_learn": sklearn.__version__},
}
recorded_history = {
    phase: {metric: [float(value) if np.isfinite(value) else None for value in values]
            for metric, values in history.items()}
    for phase, history in histories.items()
}
for filename, content in {
    "labels.json": run_config,
    "splits.json": {"audit": audit, "splits": manifest},
    "evaluation.json": {"test_report": test_report, "confusion_matrix": matrix.tolist(),
                        "history": recorded_history},
}.items():
    (RUN_DIR / filename).write_text(json.dumps(content, indent=2, allow_nan=False) + "\n")

print(f"Saved run artifacts in {RUN_DIR}")

## Predict a single image

Load an exported checkpoint and apply the same resize and padding as training. The output is the highest-scoring class and a softmax score per class. Treat these scores as uncalibrated.

In [ ]:
def predict_image(image_path, saved_run):
    saved_run = Path(saved_run)
    metadata = json.loads((saved_run / "labels.json").read_text())
    classifier = tf.keras.models.load_model(saved_run / "model.keras", compile=False)
    image = load_rgb_image(str(image_path), metadata["image_size"])
    scores = classifier(image[None, ...], training=False).numpy()[0]
    labels = metadata["class_names"]
    return {"label": labels[int(scores.argmax())], "scores": dict(zip(labels, scores.tolist()))}


# Inspect a sample prediction from the exported checkpoint.
print(predict_image(splits["test"][0][0], RUN_DIR))